# Pipeline equivalence check

Verifies that `UniqueWindowDataset` (which applies SNPs on-the-fly to the reference)
produces the same DNA windows as extracting substrings from full per-sample genomes
written by `bcftools consensus`.

**Pre-requisite:** run `make test-consensus` in the `variant_cache/` directory first.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

import pandas as pd
from pyfaidx import Fasta

from crop_embed.data.vcf import load_snps_from_vcf
from crop_embed.partitioner import SNPWindowPartitioner
from crop_embed.dataset import UniqueWindowDataset

DATA         = Path("../rice_data")
VCF_PATH     = str(DATA / "sativas413_msu7_final.vcf")
FASTA_PATH   = str(DATA / "Oryza_sativa.IRGSP-1.0.dna_sm.toplevel.fa")
TEST_DIR     = Path("test_consensus")

HALF_WINDOW  = 512
BUFFER       = 128

TEST_SAMPLES = [
    "081215-A05_1",
    "081215-A06_3",
    "081215-A07_4",
    "081215-A08_5",
    "090414-A09_6",
]

## 1. Build dataset

In [2]:
snps_by_chrom, _ = load_snps_from_vcf(VCF_PATH, TEST_SAMPLES)
partitioner = SNPWindowPartitioner(snps_by_chrom, HALF_WINDOW, BUFFER)
dataset = UniqueWindowDataset(VCF_PATH, FASTA_PATH, partitioner, TEST_SAMPLES)

print(f"Windows : {len(partitioner.windows)}")
print(f"Samples : {dataset.samples}")
print("Window stats:", partitioner.snps_per_window_stats())

Windows : 23562
Samples : ['081215-A05_1', '081215-A06_3', '081215-A07_4', '081215-A08_5', '090414-A09_6']
Window stats: {'n_windows': 23562, 'total_snps': 31076, 'mean': 1.32, 'median': 1.0, 'max': 12, 'min': 1}


## 2. Chromosome-name mapping sanity check

The IRGSP FASTA has sequence order `1, Mt, Pt, 2, 3, …`, so a naive
enumeration-based map (`{i+1: key}`) would assign chrom 2 → "Mt".
`dataset._chrom_name` uses that enumeration; `chr_to_key` below uses
the correct name-based approach from `coords.chrom_name_map`.

In [3]:
ref_fasta = Fasta(FASTA_PATH)
# Correct mapping: chromosome-number int → FASTA key string
chr_to_key = {int(k): k for k in ref_fasta.keys() if k.isdigit()}

print(f"{'chrom':>6}  {'dataset._chrom_name':>22}  {'chr_to_key (correct)':>22}  status")
print("-" * 65)
for chrom_int in sorted(snps_by_chrom.keys()):
    ds_key  = dataset._chrom_name.get(chrom_int, "MISSING")
    ok_key  = chr_to_key.get(chrom_int, "MISSING")
    status  = "OK" if ds_key == ok_key else f"BUG (got {ds_key!r})"
    print(f"{chrom_int:>6}  {ds_key:>22}  {ok_key:>22}  {status}")

 chrom     dataset._chrom_name    chr_to_key (correct)  status
-----------------------------------------------------------------
     1                       1                       1  OK
     2                       2                       2  OK
     3                       3                       3  OK
     4                       4                       4  OK
     5                       5                       5  OK
     6                       6                       6  OK
     7                       7                       7  OK
     8                       8                       8  OK
     9                       9                       9  OK
    10                      10                      10  OK
    11                      11                      11  OK
    12                      12                      12  OK


## 3. Load per-sample consensus FASTAs

In [4]:
for sample in TEST_SAMPLES:
    fa_path = TEST_DIR / f"{sample}.fa"
    assert fa_path.exists(), (
        f"Missing: {fa_path}\n"
        "Run 'make test-consensus' in variant_cache/ first."
    )

consensus = {s: Fasta(str(TEST_DIR / f"{s}.fa")) for s in TEST_SAMPLES}
print("Loaded:", list(consensus.keys()))

Loaded: ['081215-A05_1', '081215-A06_3', '081215-A07_4', '081215-A08_5', '090414-A09_6']


## 4. Compare sequences — pipeline vs bcftools consensus

For each `(sample, window)` pair we:
1. Extract the window from the pipeline (`dataset.get_item_for_sample_window`).
2. Extract the same genomic interval from the bcftools-consensus FASTA, call
   `.upper()` to strip soft-masking (matching what `_ref_seq` does), and pad
   with `N` at chromosome edges.
3. Compare the two strings.

In [5]:
def extract_from_consensus(fa: Fasta, chrom_key: str, start: int, end: int) -> str:
    """Clip to chromosome bounds, call upper(), pad edges with N."""
    chrom_len  = len(fa[chrom_key])
    clip_start = max(0, start)
    clip_end   = min(chrom_len, end)
    seq        = str(fa[chrom_key][clip_start:clip_end]).upper()
    left_pad   = max(0, -start)
    right_pad  = max(0, end - chrom_len)
    return "N" * left_pad + seq + "N" * right_pad


records = []

for sample in dataset.samples:
    con_fa = consensus[sample]
    for window in partitioner.windows:
        chrom = window.chrom
        if chrom not in chr_to_key:
            continue  # skip organelles / unplaced scaffolds

        pipe_seq = dataset.get_item_for_sample_window(sample, window.index)["sequence"]
        con_seq  = extract_from_consensus(
            con_fa, chr_to_key[chrom], window.start, window.end
        )

        records.append({
            "sample":  sample,
            "win_idx": window.index,
            "chrom":   chrom,
            "start":   window.start,
            "end":     window.end,
            "match":   pipe_seq == con_seq,
        })

df = pd.DataFrame(records)
print(df["match"].value_counts())
print(f"\nTotal comparisons : {len(df):,}")
print(f"Match rate        : {df['match'].mean():.2%}")

match
True     117806
False         4
Name: count, dtype: int64

Total comparisons : 117,810
Match rate        : 100.00%


## 5. Mismatch summary

In [6]:
mismatches = df[~df["match"]]

if mismatches.empty:
    print("All sequences match — pipelines are equivalent.")
else:
    print(f"{len(mismatches):,} mismatches out of {len(df):,}")
    print("\nMismatches by chromosome:")
    print(mismatches.groupby("chrom").size().rename("count"))
    print("\nMismatches by sample:")
    print(mismatches.groupby("sample").size().rename("count"))

4 mismatches out of 117,810

Mismatches by chromosome:
chrom
6    4
Name: count, dtype: int64

Mismatches by sample:
sample
081215-A06_3    1
081215-A07_4    1
081215-A08_5    1
090414-A09_6    1
Name: count, dtype: int64


## 6. Inspect first mismatch in detail

In [7]:
if not mismatches.empty:
    row     = mismatches.iloc[0]
    sample  = row["sample"]
    window  = partitioner.windows[int(row["win_idx"])]
    chrom   = window.chrom

    pipe_seq = dataset.get_item_for_sample_window(sample, window.index)["sequence"]
    con_seq  = extract_from_consensus(
        consensus[sample], chr_to_key[chrom], window.start, window.end
    )

    diffs = [
        (i, p, c)
        for i, (p, c) in enumerate(zip(pipe_seq, con_seq))
        if p != c
    ]

    print(f"Sample  : {sample}")
    print(f"Window  : chrom={chrom} [{window.start}, {window.end})")
    print(f"Differing positions : {len(diffs)} / {len(pipe_seq)}")
    print()
    print(f"{'offset':>8}  {'genomic_pos':>12}  {'pipeline':>10}  {'consensus':>10}")
    for offset, p, c in diffs[:20]:
        genomic = window.start + offset
        print(f"{offset:>8}  {genomic:>12}  {p:>10}  {c:>10}")
    if len(diffs) > 20:
        print(f"  … and {len(diffs) - 20} more")
else:
    print("No mismatches to inspect.")

Sample  : 081215-A06_3
Window  : chrom=6 [17161662, 17162686)
Differing positions : 1 / 1024

  offset   genomic_pos    pipeline   consensus
     128      17161790           T           A


In [9]:

# ── Diagnostic: trace the specific mismatch at chrom=1, pos=722980 ──────────
import pysam
from crop_embed.data.vcf import _parse_chrom

DIAG_CHROM  = 1
DIAG_POS    = 722980   # 0-based (VCF POS = 722981)
DIAG_SAMPLE = "081215-A05_1"
DIAG_SIDX   = dataset.samples.index(DIAG_SAMPLE)

# 1. What does pysam read for the GT? (sequential scan — plain VCF has no index)
print("=== pysam raw GT ===")
with pysam.VariantFile(VCF_PATH) as vcf:
    for rec in vcf.fetch():
        if rec.pos == DIAG_POS and _parse_chrom(rec.chrom) == DIAG_CHROM:
            gt = rec.samples[DIAG_SAMPLE]["GT"]
            print(f"  rec.pos={rec.pos}  REF={rec.ref}  ALT={rec.alts}  GT={gt}")
            print(f"  gt[0]={gt[0] if gt else None}")
            print(f"  gt_alts[{DIAG_SIDX}] would be: {1 if gt and gt[0] == 1 else 0}")
            break
    else:
        print("  !! record not found at this position")

# 2. What is stored in the partitioner's SNP list at this position?
print("\n=== partitioner snps_by_chrom[1] around pos 722980 ===")
snps_ch1 = partitioner.snps_by_chrom.get(DIAG_CHROM, [])
nearby = [r for r in snps_ch1 if abs(r.pos - DIAG_POS) <= 5]
for r in nearby:
    print(f"  pos={r.pos}  ref={chr(r.ref_byte)}  alt={chr(r.alt_byte)}  gt_alts[{DIAG_SIDX}]={r.gt_alts[DIAG_SIDX]}")

# 3. What does _alt_byte have at this position?
print(f"\n=== dataset._alt_byte[{DIAG_CHROM}].get({DIAG_POS}) ===")
ab = dataset._alt_byte.get(DIAG_CHROM, {}).get(DIAG_POS)
print(f"  alt_byte={ab}  chr={chr(ab) if ab else None}")

# 4. What are the alt_positions in the fingerprint for this sample at the mismatching window?
DIAG_WIN_START = 722439
DIAG_WIN_IDX   = next(w.index for w in partitioner.windows
                       if w.chrom == DIAG_CHROM and w.start == DIAG_WIN_START)
fp = dataset.get_fingerprint(DIAG_SAMPLE, DIAG_WIN_IDX)
print(f"\n=== fingerprint for ({DIAG_SAMPLE}, win {DIAG_WIN_IDX}) ===")
print(f"  chrom={fp[0]}  start={fp[1]}  end={fp[2]}")
print(f"  alt_positions={fp[3]}")
print(f"  {DIAG_POS} in alt_positions: {DIAG_POS in fp[3]}")


=== pysam raw GT ===
  !! record not found at this position

=== partitioner snps_by_chrom[1] around pos 722980 ===
  pos=722980  ref=C  alt=T  gt_alts[0]=1

=== dataset._alt_byte[1].get(722980) ===
  alt_byte=84  chr=T


StopIteration: 